# FinBERT news scoring

In Colab, select a GPU runtime and place `finbert_input_8149.parquet` in the configured Drive folder, or choose upload mode. For local execution, launch the notebook from the repository root and select local mode. The output retains all input columns and adds four sentiment scores per text variant.

In [ ]:
%pip install -q "transformers>=4.30,<5" pyarrow

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CFG = dict(
    DATA_SOURCE="drive",                                    # "drive" | "upload" | "local"
    DRIVE_DIR="/content/drive/MyDrive/MarketSense/finbert",
    LOCAL_DIR="data/finbert",
    MODEL_NAME="ProsusAI/finbert",
    BATCH_SIZE=32,
    MAX_LENGTH=512,
)

EXPECTED_ROWS = 8_149
INPUT_FILE = "finbert_input_8149.parquet"
OUTPUT_FILE = "finbert_scored_8149.parquet"
VARIANTS = {"hl": "text_headline", "hl_lead": "text_headline_lead"}

In [ ]:
def get_data_dir() -> Path:
    if CFG["DATA_SOURCE"] == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        return Path(CFG["DRIVE_DIR"])
    if CFG["DATA_SOURCE"] == "upload":
        from google.colab import files
        up = files.upload()                       # select finbert_input_8149.parquet
        d = Path("/content/finbert_upload")
        d.mkdir(exist_ok=True)
        for name, data in up.items():
            (d / re.sub("[ ]?[(][0-9]+[)]", "", name)).write_bytes(data)
        return d
    return Path(CFG["LOCAL_DIR"])

DATA_DIR = get_data_dir()
ds = pd.read_parquet(DATA_DIR / INPUT_FILE)
assert len(ds) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS:,} rows, got {len(ds):,}"
assert ds[list(VARIANTS.values())].notna().all().all(), "Missing input text"
print(f"Loaded {len(ds):,} articles")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime or run locally with CUDA-enabled PyTorch.")

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(CFG["MODEL_NAME"])
model = AutoModelForSequenceClassification.from_pretrained(CFG["MODEL_NAME"]).to(device).eval()
label_columns = {
    label.lower(): int(index) for index, label in model.config.id2label.items()
}
labels = {"pos": "positive", "neg": "negative", "neu": "neutral"}
assert set(label_columns) == set(labels.values()), f"Unexpected labels: {label_columns}"


@torch.inference_mode()
def score(texts: list[str], description: str) -> np.ndarray:
    batches = []
    batch_size = CFG["BATCH_SIZE"]
    for start in tqdm(range(0, len(texts), batch_size), desc=description):
        inputs = tokenizer(
            texts[start:start + batch_size],
            padding=True,
            truncation=True,
            max_length=CFG["MAX_LENGTH"],
            return_tensors="pt",
        ).to(device)
        probabilities = model(**inputs).logits.float().softmax(dim=-1)
        batches.append(probabilities.cpu().numpy())
    return np.concatenate(batches, axis=0)

In [ ]:
score_columns = []
for prefix, text_column in VARIANTS.items():
    probabilities = score(ds[text_column].tolist(), text_column)
    for suffix, label in labels.items():
        column = f"{prefix}_{suffix}"
        ds[column] = probabilities[:, label_columns[label]]
        score_columns.append(column)
    net_column = f"{prefix}_net_sentiment"
    ds[net_column] = ds[f"{prefix}_pos"] - ds[f"{prefix}_neg"]
    score_columns.append(net_column)

assert len(ds) == EXPECTED_ROWS, "Scoring changed the row count"
assert np.isfinite(ds[score_columns].to_numpy()).all(), "Non-finite sentiment scores"

output = DATA_DIR / OUTPUT_FILE
ds.to_parquet(output, index=False)
saved = pd.read_parquet(output, columns=score_columns)
assert len(saved) == EXPECTED_ROWS, "Unexpected saved row count"
assert np.isfinite(saved.to_numpy()).all(), "Non-finite saved sentiment scores"
print(f"Verified {len(saved):,} scored rows in {output}")

if CFG["DATA_SOURCE"] == "upload":
    from google.colab import files
    files.download(str(output))